In [6]:
!pip install -r requirements.txt

  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.2.1
    Uninstalling python-dotenv-1.2.1:
      Successfully uninstalled python-dotenv-1.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.20.0 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incompatible.


In [ ]:
"""
Feature Engineer Agent - Production Grade AI Agent for Automated Feature Engineering
Specializes in creating derived features from raw data to improve ML model performance.

This agent uses LangGraph orchestration with human-in-the-loop approval for
intelligent feature generation and selection.
"""

import os
import sys
import json
import logging
import warnings
from typing import TypedDict, List, Dict, Any, Optional, Literal
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END

warnings.filterwarnings('ignore')

# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('feature_engineer_agent.log'),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# ============================================================================
# ENVIRONMENT DETECTION AND API KEY LOADING
# ============================================================================

def detect_environment() -> str:
    """Detect if running in Google Colab or local environment."""
    try:
        import google.colab
        return "colab"
    except ImportError:
        return "local"

def load_api_key() -> str:
    """Load OpenAI API key based on environment."""
    env = detect_environment()

    if env == "colab":
        try:
            from google.colab import userdata
            api_key = userdata.get('OPENAI_API_KEY')
            logger.info("✓ Loaded API key from Google Colab secrets")
            return api_key
        except Exception as e:
            logger.error(f"Failed to load API key from Colab secrets: {e}")
            raise ValueError("Please add OPENAI_API_KEY to Colab Secrets (🔑 icon in left sidebar)")
    else:
        try:
            from dotenv import load_dotenv
            load_dotenv()
            api_key = os.getenv('OPENAI_API_KEY')
            if not api_key:
                raise ValueError("OPENAI_API_KEY not found in .env file")
            logger.info("✓ Loaded API key from .env file")
            return api_key
        except ImportError:
            logger.error("python-dotenv not installed. Install with: pip install python-dotenv")
            raise
        except Exception as e:
            logger.error(f"Failed to load API key from .env: {e}")
            raise

# Initialize API key
OPENAI_API_KEY = load_api_key()

# ============================================================================
# CONFIGURATION
# ============================================================================

CONFIG = {
    "max_iterations": 3,
    "min_feature_correlation": 0.05,
    "max_features_to_keep": 50,
    "human_in_loop": True,
    "target_variable": None,
    "openai_model": "gpt-4o-mini",
    "temperature": 0.7,
    "variance_threshold": 0.01
}

# ============================================================================
# STATE SCHEMA
# ============================================================================

class FeatureEngineerState(TypedDict):
    """State schema for the Feature Engineer Agent."""
    original_dataframe: Optional[pd.DataFrame]
    data_profile: Optional[Dict[str, Any]]
    feature_strategy: Optional[Dict[str, Any]]
    generated_features: Optional[Dict[str, pd.Series]]
    feature_scores: Optional[Dict[str, float]]
    selected_features: Optional[List[str]]
    transformation_pipeline: Optional[List[Dict[str, Any]]]
    user_feedback: Optional[str]
    current_iteration: int
    metadata: Dict[str, Any]

# ============================================================================
# LLM INITIALIZATION
# ============================================================================

def initialize_llm() -> ChatOpenAI:
    """Initialize OpenAI LLM with configuration."""
    return ChatOpenAI(
        model=CONFIG["openai_model"],
        temperature=CONFIG["temperature"],
        api_key=OPENAI_API_KEY
    )

# ============================================================================
# DATA PROFILING FUNCTIONS
# ============================================================================

def analyze_column_type(series: pd.Series) -> str:
    """Determine the semantic type of a column."""
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    elif pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"
    elif series.nunique() / len(series) < 0.05 and series.nunique() < 50:
        return "categorical"
    elif pd.api.types.is_string_dtype(series):
        if series.str.len().mean() > 50:
            return "text"
        return "categorical"
    else:
        return "unknown"

def profile_dataframe(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Generate comprehensive data profile for feature engineering strategy.

    Args:
        df: Input DataFrame to profile

    Returns:
        Dictionary containing data statistics and characteristics
    """
    logger.info("Profiling dataset...")

    profile = {
        "n_rows": len(df),
        "n_columns": len(df.columns),
        "columns": {},
        "missing_summary": {},
        "correlations": {}
    }

    for col in df.columns:
        col_profile = {
            "dtype": str(df[col].dtype),
            "semantic_type": analyze_column_type(df[col]),
            "missing_count": int(df[col].isnull().sum()),
            "missing_pct": float(df[col].isnull().sum() / len(df) * 100),
            "unique_count": int(df[col].nunique()),
            "unique_pct": float(df[col].nunique() / len(df) * 100)
        }

        # Add type-specific statistics
        if col_profile["semantic_type"] == "numeric":
            col_profile.update({
                "mean": float(df[col].mean()) if not df[col].isnull().all() else None,
                "std": float(df[col].std()) if not df[col].isnull().all() else None,
                "min": float(df[col].min()) if not df[col].isnull().all() else None,
                "max": float(df[col].max()) if not df[col].isnull().all() else None,
                "skewness": float(df[col].skew()) if not df[col].isnull().all() else None
            })
        elif col_profile["semantic_type"] == "categorical":
            col_profile["top_values"] = df[col].value_counts().head(5).to_dict()

        profile["columns"][col] = col_profile

    # Calculate correlations for numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 1:
        corr_matrix = df[numeric_cols].corr()
        profile["correlations"] = corr_matrix.to_dict()

    logger.info(f"✓ Profiled {len(df.columns)} columns with {len(df)} rows")
    return profile

# ============================================================================
# FEATURE ENGINEERING FUNCTIONS
# ============================================================================

def engineer_numeric_features(df: pd.DataFrame, col: str) -> Dict[str, pd.Series]:
    """Generate features from numeric columns."""
    features = {}

    try:
        series = df[col].copy()

        # Skip if all values are null
        if series.isnull().all():
            return features

        # Log transformation (for positive values)
        if (series > 0).all():
            features[f"{col}_log"] = np.log1p(series)

        # Square root (for non-negative values)
        if (series >= 0).all():
            features[f"{col}_sqrt"] = np.sqrt(series)

        # Square
        features[f"{col}_squared"] = series ** 2

        # Binning
        try:
            features[f"{col}_binned"] = pd.qcut(series, q=5, labels=False, duplicates='drop')
        except:
            pass

        # Z-score normalization
        if series.std() > 0:
            features[f"{col}_zscore"] = (series - series.mean()) / series.std()

        # Min-Max scaling
        min_val, max_val = series.min(), series.max()
        if max_val > min_val:
            features[f"{col}_minmax"] = (series - min_val) / (max_val - min_val)

    except Exception as e:
        logger.warning(f"Error engineering numeric features for {col}: {e}")

    return features

def engineer_categorical_features(df: pd.DataFrame, col: str) -> Dict[str, pd.Series]:
    """Generate features from categorical columns."""
    features = {}

    try:
        series = df[col].copy()

        # Frequency encoding
        freq_map = series.value_counts(normalize=True).to_dict()
        features[f"{col}_frequency"] = series.map(freq_map)

        # Count encoding
        count_map = series.value_counts().to_dict()
        features[f"{col}_count"] = series.map(count_map)

        # Label encoding
        le = LabelEncoder()
        features[f"{col}_label_encoded"] = pd.Series(
            le.fit_transform(series.astype(str)),
            index=series.index
        )

        # Is rare category (appears less than 5% of time)
        rare_threshold = len(df) * 0.05
        features[f"{col}_is_rare"] = series.map(count_map).apply(
            lambda x: 1 if x < rare_threshold else 0
        )

    except Exception as e:
        logger.warning(f"Error engineering categorical features for {col}: {e}")

    return features

def engineer_datetime_features(df: pd.DataFrame, col: str) -> Dict[str, pd.Series]:
    """Generate features from datetime columns."""
    features = {}

    try:
        series = pd.to_datetime(df[col], errors='coerce')

        features[f"{col}_year"] = series.dt.year
        features[f"{col}_month"] = series.dt.month
        features[f"{col}_day"] = series.dt.day
        features[f"{col}_dayofweek"] = series.dt.dayofweek
        features[f"{col}_quarter"] = series.dt.quarter
        features[f"{col}_is_weekend"] = series.dt.dayofweek.isin([5, 6]).astype(int)
        features[f"{col}_is_month_start"] = series.dt.is_month_start.astype(int)
        features[f"{col}_is_month_end"] = series.dt.is_month_end.astype(int)

        # Days since epoch
        features[f"{col}_days_since_epoch"] = (series - pd.Timestamp("1970-01-01")).dt.days

    except Exception as e:
        logger.warning(f"Error engineering datetime features for {col}: {e}")

    return features

def engineer_text_features(df: pd.DataFrame, col: str) -> Dict[str, pd.Series]:
    """Generate features from text columns."""
    features = {}

    try:
        series = df[col].astype(str)

        features[f"{col}_length"] = series.str.len()
        features[f"{col}_word_count"] = series.str.split().str.len()
        features[f"{col}_char_count"] = series.str.len()
        features[f"{col}_uppercase_count"] = series.str.count(r'[A-Z]')
        features[f"{col}_lowercase_count"] = series.str.count(r'[a-z]')
        features[f"{col}_digit_count"] = series.str.count(r'\d')
        features[f"{col}_special_char_count"] = series.str.count(r'[^a-zA-Z0-9\s]')
        features[f"{col}_whitespace_count"] = series.str.count(r'\s')

    except Exception as e:
        logger.warning(f"Error engineering text features for {col}: {e}")

    return features

def engineer_interaction_features(df: pd.DataFrame, numeric_cols: List[str]) -> Dict[str, pd.Series]:
    """Generate interaction features between numeric columns."""
    features = {}

    try:
        # Limit to top correlated pairs to avoid explosion
        max_interactions = 20
        interaction_count = 0

        for i, col1 in enumerate(numeric_cols):
            if interaction_count >= max_interactions:
                break
            for col2 in numeric_cols[i+1:]:
                if interaction_count >= max_interactions:
                    break

                try:
                    # Multiplication
                    features[f"{col1}_x_{col2}"] = df[col1] * df[col2]

                    # Division (avoid division by zero)
                    if (df[col2] != 0).all():
                        features[f"{col1}_div_{col2}"] = df[col1] / df[col2]

                    # Addition
                    features[f"{col1}_plus_{col2}"] = df[col1] + df[col2]

                    # Subtraction
                    features[f"{col1}_minus_{col2}"] = df[col1] - df[col2]

                    interaction_count += 1
                except:
                    pass

    except Exception as e:
        logger.warning(f"Error engineering interaction features: {e}")

    return features

# ============================================================================
# FEATURE EVALUATION FUNCTIONS
# ============================================================================

def calculate_feature_importance(
    df: pd.DataFrame,
    feature_name: str,
    target_col: Optional[str] = None
) -> float:
    """
    Calculate feature importance score using statistical methods.

    Args:
        df: DataFrame containing the feature
        feature_name: Name of the feature to evaluate
        target_col: Optional target variable for supervised scoring

    Returns:
        Importance score between 0 and 1
    """
    try:
        feature = df[feature_name]

        # Check variance
        if feature.var() < CONFIG["variance_threshold"]:
            return 0.0

        # If target provided, calculate correlation
        if target_col and target_col in df.columns:
            target = df[target_col]

            # For numeric target and numeric feature
            if pd.api.types.is_numeric_dtype(feature) and pd.api.types.is_numeric_dtype(target):
                corr = abs(feature.corr(target))
                if not np.isnan(corr):
                    return float(corr)

        # Default scoring based on information content
        unique_ratio = feature.nunique() / len(feature)
        missing_ratio = 1 - (feature.isnull().sum() / len(feature))

        # Prefer features with moderate uniqueness and low missing values
        score = (0.5 * (1 - abs(unique_ratio - 0.5)) + 0.5 * missing_ratio)

        return float(score)

    except Exception as e:
        logger.warning(f"Error calculating importance for {feature_name}: {e}")
        return 0.0

# ============================================================================
# LANGGRAPH NODES
# ============================================================================

def data_profiler_node(state: FeatureEngineerState) -> FeatureEngineerState:
    """
    Node 1: Profile the input dataset and extract statistical characteristics.
    """
    logger.info("=" * 80)
    logger.info("NODE 1: DATA PROFILER")
    logger.info("=" * 80)

    try:
        df = state["original_dataframe"]
        profile = profile_dataframe(df)

        state["data_profile"] = profile
        state["metadata"]["profiling_completed"] = True

        logger.info(f"✓ Data profiling completed successfully")

    except Exception as e:
        logger.error(f"Error in data profiler node: {e}")
        state["metadata"]["error"] = str(e)

    return state

def strategy_planner_node(state: FeatureEngineerState) -> FeatureEngineerState:
    """
    Node 2: Use LLM to plan feature engineering strategy based on data profile.

    System Prompt: You are a Feature Engineering Strategist with 15+ years of experience
    in machine learning and data science. Your expertise includes advanced statistical
    methods, domain knowledge across industries, and deep understanding of how different
    feature transformations impact model performance.

    User Prompt: Analyzes data profile and recommends feature engineering strategies.
    """
    logger.info("=" * 80)
    logger.info("NODE 2: STRATEGY PLANNER (LLM)")
    logger.info("=" * 80)

    try:
        llm = initialize_llm()
        profile = state["data_profile"]

        # Prepare data profile summary
        profile_summary = {
            "n_rows": profile["n_rows"],
            "n_columns": profile["n_columns"],
            "column_types": {
                col: info["semantic_type"]
                for col, info in profile["columns"].items()
            },
            "missing_data": {
                col: f"{info['missing_pct']:.1f}%"
                for col, info in profile["columns"].items()
                if info['missing_pct'] > 0
            }
        }

        system_prompt = """You are a Feature Engineering Strategist with 15+ years of experience in machine learning and data science. Your expertise includes:

- Advanced statistical methods and transformations
- Domain knowledge across multiple industries
- Deep understanding of feature engineering techniques
- Expertise in identifying patterns and relationships in data
- Knowledge of which transformations work best for different data types

Your role is to analyze data profiles and recommend optimal feature engineering strategies that will improve model performance."""

        user_prompt = f"""Analyze this dataset profile and recommend a feature engineering strategy:

Dataset Profile:
{json.dumps(profile_summary, indent=2)}

Target Variable: {CONFIG.get('target_variable', 'Not specified')}

Provide a JSON response with the following structure:
{{
    "recommended_techniques": [
        "technique 1",
        "technique 2"
    ],
    "priority_columns": [
        "column1",
        "column2"
    ],
    "interaction_candidates": [
        ["col1", "col2"]
    ],
    "reasoning": "Brief explanation of strategy"
}}

Focus on techniques that will create meaningful features without overfitting."""

        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]

        response = llm.invoke(messages)

        # Parse LLM response
        try:
            strategy = json.loads(response.content)
        except:
            # Fallback strategy
            strategy = {
                "recommended_techniques": ["numeric_transforms", "categorical_encoding"],
                "priority_columns": list(profile["columns"].keys())[:5],
                "interaction_candidates": [],
                "reasoning": "Using default strategy"
            }

        state["feature_strategy"] = strategy
        state["metadata"]["strategy_planned"] = True

        logger.info(f"✓ Strategy planned: {strategy['reasoning']}")
        logger.info(f"  Recommended techniques: {', '.join(strategy['recommended_techniques'])}")

    except Exception as e:
        logger.error(f"Error in strategy planner node: {e}")
        state["metadata"]["error"] = str(e)

    return state

def feature_generator_node(state: FeatureEngineerState) -> FeatureEngineerState:
    """
    Node 3: Generate features based on the planned strategy.
    """
    logger.info("=" * 80)
    logger.info("NODE 3: FEATURE GENERATOR")
    logger.info("=" * 80)

    try:
        df = state["original_dataframe"]
        profile = state["data_profile"]
        strategy = state["feature_strategy"]

        all_features = {}
        transformation_pipeline = []

        # Generate features for each column based on type
        for col, col_profile in profile["columns"].items():
            if col == CONFIG.get("target_variable"):
                continue

            semantic_type = col_profile["semantic_type"]

            logger.info(f"Generating features for '{col}' ({semantic_type})...")

            if semantic_type == "numeric":
                features = engineer_numeric_features(df, col)
                all_features.update(features)
                transformation_pipeline.append({
                    "column": col,
                    "type": "numeric",
                    "transformations": list(features.keys())
                })

            elif semantic_type == "categorical":
                features = engineer_categorical_features(df, col)
                all_features.update(features)
                transformation_pipeline.append({
                    "column": col,
                    "type": "categorical",
                    "transformations": list(features.keys())
                })

            elif semantic_type == "datetime":
                features = engineer_datetime_features(df, col)
                all_features.update(features)
                transformation_pipeline.append({
                    "column": col,
                    "type": "datetime",
                    "transformations": list(features.keys())
                })

            elif semantic_type == "text":
                features = engineer_text_features(df, col)
                all_features.update(features)
                transformation_pipeline.append({
                    "column": col,
                    "type": "text",
                    "transformations": list(features.keys())
                })

        # Generate interaction features
        numeric_cols = [
            col for col, info in profile["columns"].items()
            if info["semantic_type"] == "numeric" and col != CONFIG.get("target_variable")
        ]

        if len(numeric_cols) >= 2:
            logger.info("Generating interaction features...")
            interaction_features = engineer_interaction_features(df, numeric_cols[:5])
            all_features.update(interaction_features)
            transformation_pipeline.append({
                "type": "interactions",
                "transformations": list(interaction_features.keys())
            })

        state["generated_features"] = all_features
        state["transformation_pipeline"] = transformation_pipeline
        state["metadata"]["features_generated"] = len(all_features)

        logger.info(f"✓ Generated {len(all_features)} new features")

    except Exception as e:
        logger.error(f"Error in feature generator node: {e}")
        state["metadata"]["error"] = str(e)

    return state

def feature_evaluator_node(state: FeatureEngineerState) -> FeatureEngineerState:
    """
    Node 4: Evaluate and score generated features.
    """
    logger.info("=" * 80)
    logger.info("NODE 4: FEATURE EVALUATOR")
    logger.info("=" * 80)

    try:
        df = state["original_dataframe"]
        generated_features = state["generated_features"]
        target_col = CONFIG.get("target_variable")

        # Create temporary DataFrame with generated features
        temp_df = df.copy()
        for feat_name, feat_series in generated_features.items():
            temp_df[feat_name] = feat_series

        # Calculate importance scores
        feature_scores = {}
        for feat_name in generated_features.keys():
            score = calculate_feature_importance(temp_df, feat_name, target_col)
            feature_scores[feat_name] = score

        # Sort by score
        sorted_features = sorted(
            feature_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )

        state["feature_scores"] = dict(sorted_features)
        state["metadata"]["features_evaluated"] = len(feature_scores)

        # Log top features
        logger.info(f"✓ Evaluated {len(feature_scores)} features")
        logger.info("Top 10 features by importance:")
        for feat_name, score in sorted_features[:10]:
            logger.info(f"  {feat_name}: {score:.4f}")

    except Exception as e:
        logger.error(f"Error in feature evaluator node: {e}")
        state["metadata"]["error"] = str(e)

    return state

def human_review_node(state: FeatureEngineerState) -> FeatureEngineerState:
    """
    Node 5: Present features to human for approval (if enabled).
    """
    logger.info("=" * 80)
    logger.info("NODE 5: HUMAN REVIEW")
    logger.info("=" * 80)

    try:
        if not CONFIG["human_in_loop"]:
            logger.info("Human review disabled, auto-approving features")
            state["user_feedback"] = "approved"
            return state

        feature_scores = state["feature_scores"]

        # Show top features
        print("\n" + "=" * 80)
        print("FEATURE REVIEW - Top 20 Generated Features")
        print("=" * 80)

        top_features = list(feature_scores.items())[:20]
        for i, (feat_name, score) in enumerate(top_features, 1):
            print(f"{i:2d}. {feat_name:50s} Score: {score:.4f}")

        print("\n" + "=" * 80)
        print(f"Total features generated: {len(feature_scores)}")
        print(f"Features above correlation threshold ({CONFIG['min_feature_correlation']}): "
              f"{sum(1 for s in feature_scores.values() if s >= CONFIG['min_feature_correlation'])}")
        print("=" * 80)

        # Get user feedback
        print("\nOptions:")
        print("1. APPROVE - Accept and select top features")
        print("2. REJECT - Regenerate with different strategy")
        print("3. CUSTOM - Manually select features")

        feedback = input("\nYour choice (1/2/3): ").strip()

        if feedback == "1":
            state["user_feedback"] = "approved"
            logger.info("✓ Features approved by user")
        elif feedback == "2":
            state["user_feedback"] = "rejected"
            logger.info("✓ Features rejected, will regenerate")
        elif feedback == "3":
            state["user_feedback"] = "custom"
            logger.info("✓ Custom selection mode")
        else:
            state["user_feedback"] = "approved"
            logger.info("✓ Invalid input, defaulting to approval")

    except Exception as e:
        logger.error(f"Error in human review node: {e}")
        state["user_feedback"] = "approved"

    return state

def feature_selector_node(state: FeatureEngineerState) -> FeatureEngineerState:
    """
    Node 6: Select final features based on scores and user feedback.
    """
    logger.info("=" * 80)
    logger.info("NODE 6: FEATURE SELECTOR")
    logger.info("=" * 80)

    try:
        feature_scores = state["feature_scores"]
        feedback = state["user_feedback"]

        if feedback == "custom":
            # Manual selection
            print("\nEnter feature numbers to keep (comma-separated, e.g., 1,3,5,7):")
            selection = input("Features: ").strip()

            try:
                indices = [int(x.strip()) - 1 for x in selection.split(",")]
                feature_list = list(feature_scores.keys())
                selected = [feature_list[i] for i in indices if 0 <= i < len(feature_list)]
            except:
                logger.warning("Invalid selection, using automatic selection")
                selected = [
                    feat for feat, score in feature_scores.items()
                    if score >= CONFIG["min_feature_correlation"]
                ][:CONFIG["max_features_to_keep"]]
        else:
            # Automatic selection
            selected = [
                feat for feat, score in feature_scores.items()
                if score >= CONFIG["min_feature_correlation"]
            ][:CONFIG["max_features_to_keep"]]

        state["selected_features"] = selected
        state["metadata"]["features_selected"] = len(selected)

        logger.info(f"✓ Selected {len(selected)} features for final dataset")

    except Exception as e:
        logger.error(f"Error in feature selector node: {e}")
        state["metadata"]["error"] = str(e)

    return state

def output_formatter_node(state: FeatureEngineerState) -> FeatureEngineerState:
    """
    Node 7: Format and prepare final output with documentation.

    System Prompt: You are a Technical Documentation Specialist with expertise in
    data science and machine learning. You excel at creating clear, comprehensive
    documentation that helps data scientists understand feature engineering decisions.

    User Prompt: Creates feature documentation explaining each selected feature.
    """
    logger.info("=" * 80)
    logger.info("NODE 7: OUTPUT FORMATTER (LLM)")
    logger.info("=" * 80)

    try:
        llm = initialize_llm()

        df = state["original_dataframe"]
        selected_features = state["selected_features"]
        feature_scores = state["feature_scores"]
        transformation_pipeline = state["transformation_pipeline"]

        # Create enhanced dataset
        enhanced_df = df.copy()
        generated_features = state["generated_features"]

        for feat_name in selected_features:
            if feat_name in generated_features:
                enhanced_df[feat_name] = generated_features[feat_name]

        # Prepare feature summary for LLM
        feature_summary = []
        for feat in selected_features[:20]:  # Top 20 for documentation
            feature_summary.append({
                "name": feat,
                "score": feature_scores.get(feat, 0.0),
                "type": "engineered"
            })

        system_prompt = """You are a Technical Documentation Specialist with 15+ years of experience in data science and machine learning. Your expertise includes:

- Creating clear, comprehensive technical documentation
- Explaining complex feature engineering concepts
- Translating statistical methods into business insights
- Writing documentation that serves both technical and non-technical audiences

Your role is to create feature documentation that helps data scientists and stakeholders understand the feature engineering process and decisions."""

        user_prompt = f"""Create comprehensive documentation for these engineered features:

Selected Features ({len(selected_features)} total):
{json.dumps(feature_summary, indent=2)}

Transformation Pipeline:
{json.dumps(transformation_pipeline, indent=2)}

Provide a JSON response with the following structure:
{{
    "summary": "Overall summary of feature engineering process",
    "total_features_generated": number,
    "total_features_selected": number,
    "original_columns": number,
    "final_columns": number,
    "top_features": [
        {{
            "name": "feature_name",
            "score": 0.xx,
            "description": "what it represents"
        }}
    ],
    "transformation_types": {{
        "numeric": number,
        "categorical": number,
        "datetime": number,
        "text": number,
        "interaction": number
    }},
    "recommendations": "Key recommendations for using these features",
    "quality_assessment": "Overall quality of generated features"
}}

Keep descriptions concise and actionable."""

        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]

        response = llm.invoke(messages)

        # Parse documentation
        try:
            documentation = json.loads(response.content)
        except:
            # Fallback documentation
            documentation = {
                "summary": "Feature engineering completed successfully",
                "total_features_generated": len(state["generated_features"]),
                "total_features_selected": len(selected_features),
                "original_columns": len(df.columns),
                "final_columns": len(enhanced_df.columns),
                "top_features": [
                    {"name": feat, "score": feature_scores.get(feat, 0.0), "description": "Engineered feature"}
                    for feat in selected_features[:10]
                ],
                "transformation_types": {
                    "numeric": sum(1 for t in transformation_pipeline if t.get("type") == "numeric"),
                    "categorical": sum(1 for t in transformation_pipeline if t.get("type") == "categorical"),
                    "datetime": sum(1 for t in transformation_pipeline if t.get("type") == "datetime"),
                    "text": sum(1 for t in transformation_pipeline if t.get("type") == "text"),
                    "interaction": sum(1 for t in transformation_pipeline if t.get("type") == "interactions")
                },
                "recommendations": "Use these features in your ML model for improved performance",
                "quality_assessment": "Features have been selected based on statistical relevance"
            }

        # Add metadata to state
        state["metadata"]["enhanced_df"] = enhanced_df
        state["metadata"]["documentation"] = documentation
        state["metadata"]["output_ready"] = True

        logger.info(f"✓ Output formatted with {len(enhanced_df.columns)} total columns")
        logger.info(f"  Original columns: {len(df.columns)}")
        logger.info(f"  Engineered columns: {len(selected_features)}")

    except Exception as e:
        logger.error(f"Error in output formatter node: {e}")
        state["metadata"]["error"] = str(e)

    return state

# ============================================================================
# ROUTING LOGIC
# ============================================================================

def should_regenerate(state: FeatureEngineerState) -> Literal["regenerate", "continue"]:
    """Determine if features should be regenerated based on feedback."""
    feedback = state.get("user_feedback", "approved")
    iteration = state.get("current_iteration", 1)

    if feedback == "rejected" and iteration < CONFIG["max_iterations"]:
        state["current_iteration"] = iteration + 1
        logger.info(f"Regenerating features (iteration {state['current_iteration']})...")
        return "regenerate"

    return "continue"

# ============================================================================
# LANGGRAPH CONSTRUCTION
# ============================================================================

def create_feature_engineer_graph() -> StateGraph:
    """Create and configure the LangGraph for feature engineering."""

    workflow = StateGraph(FeatureEngineerState)

    # Add nodes
    workflow.add_node("data_profiler", data_profiler_node)
    workflow.add_node("strategy_planner", strategy_planner_node)
    workflow.add_node("feature_generator", feature_generator_node)
    workflow.add_node("feature_evaluator", feature_evaluator_node)
    workflow.add_node("human_review", human_review_node)
    workflow.add_node("feature_selector", feature_selector_node)
    workflow.add_node("output_formatter", output_formatter_node)

    # Define edges
    workflow.set_entry_point("data_profiler")
    workflow.add_edge("data_profiler", "strategy_planner")
    workflow.add_edge("strategy_planner", "feature_generator")
    workflow.add_edge("feature_generator", "feature_evaluator")
    workflow.add_edge("feature_evaluator", "human_review")

    # Conditional routing after human review
    workflow.add_conditional_edges(
        "human_review",
        should_regenerate,
        {
            "regenerate": "strategy_planner",
            "continue": "feature_selector"
        }
    )

    workflow.add_edge("feature_selector", "output_formatter")
    workflow.add_edge("output_formatter", END)

    return workflow.compile()

# ============================================================================
# FILE LOADING FUNCTIONS
# ============================================================================

def upload_and_load_file():
    """
    Handle file upload in both Colab and local environments.

    Returns:
        tuple: (DataFrame, file_format, original_filename)
    """
    env = detect_environment()

    if env == "colab":
        # Google Colab file upload
        print("\n" + "=" * 80)
        print("FILE UPLOAD - Click 'Choose Files' button to upload your data")
        print("Supported formats: CSV, Excel (.xlsx, .xls), JSON")
        print("=" * 80)

        try:
            from google.colab import files
            uploaded = files.upload()

            if not uploaded:
                raise ValueError("No file uploaded")

            # Get the uploaded file
            filename = list(uploaded.keys())[0]
            file_content = uploaded[filename]

            logger.info(f"File uploaded: {filename}")

            # Detect file format and load
            if filename.endswith('.csv'):
                df = pd.read_csv(filename)
                file_format = 'csv'
                logger.info("✓ Loaded CSV file")
            elif filename.endswith(('.xlsx', '.xls')):
                df = pd.read_excel(filename)
                file_format = 'excel'
                logger.info("✓ Loaded Excel file")
            elif filename.endswith('.json'):
                df = pd.read_json(filename)
                file_format = 'json'
                logger.info("✓ Loaded JSON file")
            else:
                raise ValueError(f"Unsupported file format. Please upload CSV, Excel, or JSON file.")

            # Validate DataFrame
            if df.empty:
                raise ValueError("Uploaded file contains no data")

            if len(df.columns) == 0:
                raise ValueError("Uploaded file has no columns")

            logger.info(f"✓ Loaded {len(df)} rows and {len(df.columns)} columns")

            return df, file_format, filename

        except ImportError:
            logger.error("google.colab.files not available")
            raise
        except Exception as e:
            logger.error(f"Error uploading file: {e}")
            raise

    else:
        # Local environment - use file path input
        print("\n" + "=" * 80)
        print("FILE UPLOAD")
        print("Supported formats: CSV, Excel (.xlsx, .xls), JSON")
        print("=" * 80)

        file_path = input("\nEnter the full path of your data file: ").strip()

        # Remove quotes if present
        file_path = file_path.strip('"').strip("'")

        # Validate file exists
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")

        filename = os.path.basename(file_path)
        logger.info(f"Loading file: {file_path}")

        # Detect file format and load
        try:
            if file_path.endswith('.csv'):
                df = pd.read_csv(file_path)
                file_format = 'csv'
                logger.info("✓ Loaded CSV file")
            elif file_path.endswith(('.xlsx', '.xls')):
                df = pd.read_excel(file_path)
                file_format = 'excel'
                logger.info("✓ Loaded Excel file")
            elif file_path.endswith('.json'):
                df = pd.read_json(file_path)
                file_format = 'json'
                logger.info("✓ Loaded JSON file")
            else:
                raise ValueError(f"Unsupported file format. Please provide CSV, Excel, or JSON file.")

            # Validate DataFrame
            if df.empty:
                raise ValueError("File contains no data")

            if len(df.columns) == 0:
                raise ValueError("File has no columns")

            logger.info(f"✓ Loaded {len(df)} rows and {len(df.columns)} columns")

            return df, file_format, filename

        except Exception as e:
            logger.error(f"Error loading file: {e}")
            raise

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def save_output_files(enhanced_df: pd.DataFrame,
                      documentation: Dict,
                      transformation_pipeline: List[Dict],
                      file_format: str,
                      original_filename: str) -> Dict[str, str]:
    """
    Save output files in the same format as input.

    Args:
        enhanced_df: Enhanced DataFrame with engineered features
        documentation: Feature documentation dictionary
        transformation_pipeline: List of transformations applied
        file_format: Format of input file (csv, excel, json)
        original_filename: Original filename

    Returns:
        Dictionary with output file paths
    """
    # Generate output filenames
    base_name = os.path.splitext(original_filename)[0]

    output_files = {}

    # Save enhanced dataset in same format as input
    if file_format == 'csv':
        data_file = f"{base_name}_enhanced.csv"
        enhanced_df.to_csv(data_file, index=False)
        output_files['data'] = data_file
        logger.info(f"✓ Saved enhanced dataset to: {data_file}")

    elif file_format == 'excel':
        data_file = f"{base_name}_enhanced.xlsx"
        enhanced_df.to_excel(data_file, index=False, engine='openpyxl')
        output_files['data'] = data_file
        logger.info(f"✓ Saved enhanced dataset to: {data_file}")

    elif file_format == 'json':
        data_file = f"{base_name}_enhanced.json"
        enhanced_df.to_json(data_file, orient='records', indent=2)
        output_files['data'] = data_file
        logger.info(f"✓ Saved enhanced dataset to: {data_file}")

    # Always save summary report as JSON
    report_file = f"{base_name}_feature_report.json"

    # Create comprehensive report
    report = {
        "timestamp": datetime.now().isoformat(),
        "input_file": original_filename,
        "input_format": file_format,
        "output_file": data_file,
        "feature_engineering_summary": documentation,
        "transformation_pipeline": transformation_pipeline,
        "statistics": {
            "original_rows": len(enhanced_df),
            "original_columns": documentation.get("original_columns", 0),
            "engineered_columns": documentation.get("total_features_selected", 0),
            "total_columns": documentation.get("final_columns", 0)
        }
    }

    with open(report_file, "w") as f:
        json.dump(report, f, indent=2)

    output_files['report'] = report_file
    logger.info(f"✓ Saved feature report to: {report_file}")

    return output_files

def main():
    """Main execution function for the Feature Engineer Agent."""

    print("\n" + "=" * 80)
    print("FEATURE ENGINEER AGENT")
    print("Automated Feature Engineering with LangGraph and GPT-4o-mini")
    print("=" * 80)

    try:
        # Step 1: Upload and load data
        df, file_format, original_filename = upload_and_load_file()

        # Step 2: Display data preview
        print("\n" + "=" * 80)
        print("DATA PREVIEW")
        print("=" * 80)
        print(df.head())
        print(f"\nShape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        print(f"File Format: {file_format.upper()}")

        # Step 3: Get target variable
        print("\n" + "=" * 80)
        print("TARGET VARIABLE SELECTION (Optional)")
        print("=" * 80)
        print("\nAvailable columns:")
        for i, col in enumerate(df.columns, 1):
            print(f"{i}. {col}")

        target_input = input("\nEnter target variable name (or press Enter to skip): ").strip()

        if target_input and target_input in df.columns:
            CONFIG["target_variable"] = target_input
            logger.info(f"Target variable set to: {target_input}")
        else:
            logger.info("No target variable specified (unsupervised mode)")

        # Step 4: Configure human-in-the-loop
        human_loop = input("\nEnable human-in-the-loop review? (y/n, default=y): ").strip().lower()
        CONFIG["human_in_loop"] = human_loop != "n"

        # Step 5: Initialize state
        initial_state = FeatureEngineerState(
            original_dataframe=df,
            data_profile=None,
            feature_strategy=None,
            generated_features=None,
            feature_scores=None,
            selected_features=None,
            transformation_pipeline=None,
            user_feedback=None,
            current_iteration=1,
            metadata={
                "start_time": datetime.now().isoformat(),
                "config": CONFIG.copy(),
                "input_file": original_filename,
                "input_format": file_format
            }
        )

        # Step 6: Create and run graph
        logger.info("\n" + "=" * 80)
        logger.info("INITIALIZING LANGGRAPH")
        logger.info("=" * 80)

        graph = create_feature_engineer_graph()

        logger.info("\n" + "=" * 80)
        logger.info("EXECUTING FEATURE ENGINEERING PIPELINE")
        logger.info("=" * 80)

        final_state = graph.invoke(initial_state)

        # Step 7: Save outputs in same format as input
        if final_state["metadata"].get("output_ready"):
            enhanced_df = final_state["metadata"]["enhanced_df"]
            documentation = final_state["metadata"]["documentation"]
            transformation_pipeline = final_state["transformation_pipeline"]

            output_files = save_output_files(
                enhanced_df=enhanced_df,
                documentation=documentation,
                transformation_pipeline=transformation_pipeline,
                file_format=file_format,
                original_filename=original_filename
            )

            print("\n" + "=" * 80)
            print("✅ FEATURE ENGINEERING COMPLETED SUCCESSFULLY!")
            print("=" * 80)
            print(f"\nOriginal features: {len(df.columns)}")
            print(f"Engineered features: {len(final_state['selected_features'])}")
            print(f"Total features: {len(enhanced_df.columns)}")
            print(f"\n📁 Output Files (Same format as input: {file_format.upper()}):")
            print(f"  📊 Enhanced Data: {output_files['data']}")
            print(f"  📋 Feature Report: {output_files['report']}")
            print("\n" + "=" * 80)

            # Display summary from report
            print("\n📈 FEATURE ENGINEERING SUMMARY:")
            print("=" * 80)
            doc = documentation
            print(f"Summary: {doc.get('summary', 'N/A')}")
            print(f"\nTotal Features Generated: {doc.get('total_features_generated', 0)}")
            print(f"Features Selected: {doc.get('total_features_selected', 0)}")
            print(f"\nRecommendations: {doc.get('recommendations', 'N/A')}")
            print(f"Quality Assessment: {doc.get('quality_assessment', 'N/A')}")
            print("=" * 80)

        else:
            logger.error("Feature engineering pipeline did not complete successfully")
            if "error" in final_state["metadata"]:
                logger.error(f"Error: {final_state['metadata']['error']}")

    except KeyboardInterrupt:
        print("\n\n⚠️  Process interrupted by user")
        logger.info("Process interrupted by user")
    except Exception as e:
        logger.error(f"Fatal error in main execution: {e}", exc_info=True)
        print(f"\n❌ Error: {e}")
        print("Check feature_engineer_agent.log for details")

if __name__ == "__main__":
    main()


FEATURE ENGINEER AGENT
Automated Feature Engineering with LangGraph and GPT-4o-mini

FILE UPLOAD - Click 'Choose Files' button to upload your data
Supported formats: CSV, Excel (.xlsx, .xls), JSON


Saving sample_customer_data.csv to sample_customer_data (1).csv

DATA PREVIEW
  customer_id signup_date last_purchase_date  total_purchases  total_spent  \
0  CUST_00001  2022-03-15         2024-11-28               45      2340.50   
1  CUST_00002  2022-06-22         2024-10-15               12       890.25   
2  CUST_00003  2023-01-10         2024-12-05               67      4523.80   
3  CUST_00004  2021-11-03         2024-03-20                8       234.60   
4  CUST_00005  2023-05-18         2024-11-22               23      1567.30   

   avg_order_value favorite_category  days_since_last_purchase  \
0            52.01       Electronics                        13   
1            74.19          Clothing                        57   
2            67.52     Home & Garden                         6   
3            29.33             Books                       265   
4            68.14            Sports                        19   

   email_opened_last_30_days  customer_age  ... newslett